# Real LLM client smoke test

In [ ]:
import json
import logging
import sys
from pathlib import Path
from time import perf_counter

from dotenv import load_dotenv


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "utils" / "llm_client.py").is_file():
            return candidate
    raise FileNotFoundError("Could not locate the project root")


project_root = find_project_root(Path.cwd().resolve())
env_path = project_root / ".env"
load_dotenv(env_path, override=False)
sys.path.insert(0, str(project_root))

from utils.llm_client import LLMClient

print(f"Project root: {project_root}")
print(f"Environment file: {env_path}")

In [ ]:
llm_client = LLMClient()

print(f"Model: {llm_client.config.model}")
print(f"Base URL: {llm_client.config.base_url or 'OpenAI SDK default'}")
print(f"API key loaded: {bool(llm_client.config.api_key)}")

In [ ]:
messages = [
    {
        "role": "system",
        "content": "Return only a valid JSON object.",
    },
    {
        "role": "user",
        "content": (
            "Classify this shopping request: 'I need black running shoes under $100.' "
            "Return JSON with intent, category, color, and budget_max fields."
        ),
    },
]

started_at = perf_counter()
try:
    result = llm_client.generate_json(messages)
except Exception:
    logging.exception("failed to generate JSON response")
    raise
elapsed_seconds = perf_counter() - started_at

print("\nParsed JSON response:")
print(json.dumps(result, indent=2, ensure_ascii=False))
print(f"\nElapsed time: {elapsed_seconds:.3f} seconds")
print("Latest call usage:", llm_client.last_usage.as_dict())
print("Latest call total tokens:", llm_client.last_usage.total_tokens)
print("Cumulative usage:", llm_client.total_usage.as_dict())
print("Usage reportable by Agent:", llm_client.consume_usage().as_dict())